# 📘 Semaine 11 — Communication : Exposés UART/USART et SMBus

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 exposés + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Présenter** le bus UART/USART (asynchrone vs synchrone, trame, baud rate).
2. **Présenter** le bus SMBus (variante I2C, PEC, timeout).
3. **Comparer** les 4 bus étudiés (UART, I2C, SPI, SMBus).
4. **Configurer** USART2 et communiquer avec un PC.
5. **Mettre en œuvre** une communication bidirectionnelle (commandes du PC).

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Exposés** | Groupes 3 (UART) et 4 (SMBus) + synthèse | 1h30 |
| **B — Atelier** | TP11 : UART ↔ PC + démo SMBus | 1h30 |
| **C — Homework** | Rapport comparatif + préparation S12 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — EXPOSÉS ÉTUDIANTS (1h30)

## 🔹 Organisation

### Groupes et thématiques

| Groupe | Thème | Durée |
|---|---|---|
| **Groupe 3** | UART / USART | 20 min + 5 min Q |
| **Groupe 4** | SMBus | 20 min + 5 min Q |
| **Enseignant** | Synthèse comparative des 4 bus | 20 min |

### 📋 Grille d'évaluation de l'exposé

| Critère | 0-2 pts |
|---|---|
| Clarté du plan | |
| Exactitude technique | |
| Support visuel | |
| Réponses aux questions | |
| Respect du temps | |
| **Total** | **/10** |

---

## 🔹 Exposé 3 — UART / USART

### 📝 Plan attendu (groupe 3)

1. **Historique** — Universal Asynchronous Receiver Transmitter (années 60)
2. **UART vs USART**
   - UART : asynchrone uniquement
   - USART : asynchrone + synchrone
3. **Lignes physiques**
   - **TX** : transmission
   - **RX** : réception
   - **RTS/CTS** (optionnel) : flow control
4. **Trame asynchrone**
   - Start bit (1)
   - Data (7/8/9 bits)
   - Parity (none/even/odd)
   - Stop bit (1/2)
5. **Baud rate et génération**
   - USART_BRR = F_clk / baud
   - Baud rates standard : 9600, 19200, 115200...
6. **Modes**
   - Half-duplex, Full-duplex
   - Multi-processeur
   - IrDA, LIN, Smartcard
7. **Applications** — console série, GPS, Bluetooth, Modbus, RS-232, RS-485

### 📖 Fiche de référence UART/USART

#### Trame UART 8N1

```
Idle      Start    D0   D1   D2   D3   D4   D5   D6   D7   Stop    Idle
  ───┐    ┌───┬───┬───┬───┬───┬───┬───┬───┬───┬────┐
     │    │   │   │   │   │   │   │   │   │   │    │
     └────┘   └───┴───┴───┴───┴───┴───┴───┴───┘    └────
        1 bit  8 bits de données (LSB first) 1-2 bits
```

#### Format de trame

| Élément | Bits | État |
|---|---|---|
| **Idle** | — | 1 (haut) |
| **Start** | 1 | 0 (bas) |
| **Data** | 7/8/9 | LSB first |
| **Parity** | 0/1 | Even/Odd/None |
| **Stop** | 1/2 | 1 (haut) |

#### Notations usuelles

| Notation | Signification |
|---|---|
| 8N1 | 8 data, no parity, 1 stop |
| 8E1 | 8 data, even parity, 1 stop |
| 8O1 | 8 data, odd parity, 1 stop |
| 7E1 | 7 data, even parity, 1 stop |

#### Baud rate sur STM32F103

```
USARTDIV = F_clk / (16 × baud)
USART_BRR = mantisse (12 bits) + fraction (4 bits)
```

**Exemple :** F_clk = 72 MHz, baud = 115200 :
```
USARTDIV = 72 000 000 / (16 × 115200) = 39.0625
Mantisse = 39 = 0x27
Fraction = 0.0625 × 16 = 1
USART_BRR = 0x271
```

#### RTS / CTS (flow control matériel)

| Signal | Direction | Rôle |
|---|---|---|
| **RTS** | Émetteur → Récepteur | Prêt à envoyer |
| **CTS** | Récepteur → Émetteur | Prêt à recevoir |

### 🐍 Simulation Python — Génération d'une trame UART (10 min)

In [ ]:
# ============================================================
# Génération d'une trame UART (8N1)
# ============================================================

def generer_trame_uart(octet, parity='none', stop_bits=1):
    """
    Génère la trame UART d'un octet.
    Retourne la liste des bits : [start, d0..d7, parity?, stop...].
    """
    bits = [0]                                  # Start bit
    for i in range(8):
        bits.append((octet >> i) & 1)          # LSB first
    if parity == 'even':
        bits.append(sum(bits[1:9]) % 2)
    elif parity == 'odd':
        bits.append(1 - (sum(bits[1:9]) % 2))
    bits += [1] * stop_bits                    # Stop bit(s)
    return bits

def afficher_trame(bits, titre=""):
    if titre:
        print(f"\n📡 {titre}")
    for i, b in enumerate(bits):
        symbole = "█" if b else "_"
        print(f"  Bit {i:<2} : {b}  {symbole}")

# Exemple : envoyer 'A' (0x41 = 65)
octet = ord('A')
trame = generer_trame_uart(octet, parity='none', stop_bits=1)

print(f"📤 Envoi du caractère 'A' (0x{octet:02X} = 0b{octet:08b})\n")
print(f"Trame 8N1 ({len(trame)} bits) :")
print(f"  [start] {' '.join(str(b) for b in trame[1:9])} [stop]")

print(f"\n  Start : {trame[0]}")
print(f"  Data  : {''.join(str(b) for b in trame[1:9])} (LSB first)")
print(f"  Stop  : {trame[-1]}")

### 🐍 Calcul du registre BRR (10 min)

In [ ]:
# ============================================================
# Calcul du registre USART_BRR
# ============================================================

def calcul_brr(f_clk_hz, baud, oversampling=16):
    """
    Calcule la valeur du registre USART_BRR.
    - oversampling : 16 (standard) ou 8 (OVER8)
    Retourne (brr, mantisse, fraction, erreur_pct).
    """
    usartdiv = f_clk_hz / (oversampling * baud)
    mantisse = int(usartdiv)
    fraction = usartdiv - mantisse
    if oversampling == 16:
        frac_int = round(fraction * 16)
        brr = (mantisse << 4) | (frac_int & 0x0F)
    else:
        frac_int = round(fraction * 8)
        brr = (mantisse << 4) | (frac_int & 0x07)
    baud_reel = f_clk_hz / (oversampling * (mantisse + frac_int / oversampling))
    err = abs(baud_reel - baud) / baud * 100
    return brr, mantisse, frac_int, err

F_CLK = 72_000_000

print(f"📊 Calcul USART_BRR — F_clk = {F_CLK/1e6:.0f} MHz\n")
print(f"{'Baud':<10}{'BRR':<10}{'Mantisse':<12}{'Frac':<8}{'Erreur %'}")
print("-" * 55)
for baud in [9600, 19200, 38400, 57600, 115200, 230400, 460800, 921600]:
    brr, m, f, err = calcul_brr(F_CLK, baud)
    print(f"{baud:<10}0x{brr:04X}    {m:<12}{f:<8}{err:.4f}")

print(f"\n📌 À retenir :")
print(f"  → Erreur < 2 % est acceptable.")
print(f"  → Au-delà, risque de trames corrompues.")

---

## 🔹 Exposé 4 — SMBus

### 📝 Plan attendu (groupe 4)

1. **Historique** — System Management Bus, spécifié par **Intel** (1995)
2. **Principe** — variante du **I2C** pour la gestion de systèmes
3. **Différences avec I2C**
   - **Timeout** obligatoire (35 ms) : évite les blocages
   - **PEC** (Packet Error Checking) optionnel
   - Niveaux logiques plus stricts (V_IH = 2.1 V)
   - **Fréquence** : 10 kHz à 100 kHz
4. **Protocole**
   - Basé sur I2C (START, STOP, ACK)
   - Adresses réservées (SMBus Host, Alert, etc.)
   - Commandes standardisées
5. **PEC (Packet Error Checking)**
   - CRC-8 calculé sur toute la trame
   - Polygone : x^8 + x^2 + x + 1 (0x07)
6. **Applications**
   - Gestion de batterie (Smart Battery)
   - Capteurs de température (SPD EEPROM)
   - Contrôleurs d'alimentation
   - ACPI (Advanced Configuration and Power Interface)

### 📖 Fiche de référence SMBus

#### Comparaison I2C vs SMBus

| Critère | I2C | SMBus |
|---|---|---|
| **Débit minimum** | 0 | **10 kHz** |
| **Débit maximum** | 1 MHz (Fast+) | 100 kHz |
| **Timeout** | Non | **35 ms obligatoire** |
| **PEC** | Non | **Optionnel** |
| **V_IH** | 0.7 × VDD | **2.1 V fixe** |
| **Adresses réservées** | 16 | **Plus de 20** |
| **Alert line** | Non | **SMBALERT#** |
| **Applications** | Général | Gestion système |

#### Adresses SMBus réservées (extrait)

| Adresse | Usage |
|---|---|
| 0x08 | SMBus Host |
| 0x0C | SMBus Alert Response |
| 0x28 | ACCESS.bus |
| 0x50-0x57 | EEPROM SPD (RAM) |
| 0x61 | SMBus Device Default |

#### PEC — Packet Error Checking

**Calcul CRC-8 :**
```
Polynôme : x^8 + x^2 + x + 1 = 0x07
Initial : 0x00
Bit de poids fort traité en premier
```

Le PEC est **ajouté en fin de trame** et vérifié par l'esclave.

### 🐍 Simulation Python — CRC-8 PEC (15 min)

In [ ]:
# ============================================================
# Calcul du PEC SMBus (CRC-8)
# ============================================================

def crc8_smbus(donnees):
    """
    Calcule le CRC-8 SMBus.
    - donnees : liste d'octets
    - Polygone : 0x07 (x^8 + x^2 + x + 1)
    """
    crc = 0x00
    for octet in donnees:
        crc ^= octet
        for _ in range(8):
            if crc & 0x80:
                crc = ((crc << 1) ^ 0x07) & 0xFF
            else:
                crc = (crc << 1) & 0xFF
    return crc

# Exemple de trame SMBus
adresse  = 0x50
commande = 0x10
donnee   = 0xAB

trame = [adresse << 1, commande, donnee]   # W=0
pec = crc8_smbus(trame)

print("📡 Trame SMBus avec PEC\n")
print(f"  Adresse esclave : 0x{adresse:02X}")
print(f"  Commande        : 0x{commande:02X}")
print(f"  Donnée          : 0x{donnee:02X}")
print(f"  Octets pour PEC : {[f'0x{b:02X}' for b in trame]}")
print(f"  PEC calculé     : 0x{pec:02X}")
print(f"\n  Trame complète  : [S] {' '.join(f'{b:02X}' for b in trame)} {pec:02X} [P]")

# Test de détection d'erreur
print("\n🔍 Test de détection d'erreur :")
trame_err = [adresse << 1, commande, donnee ^ 0x01]
pec_err = crc8_smbus(trame_err)
print(f"  Trame avec erreur : {[f'0x{b:02X}' for b in trame_err]}")
print(f"  PEC recalculé     : 0x{pec_err:02X}  (≠ 0x{pec:02X}, erreur détectée)")

---

## 🔹 Synthèse comparative — Les 4 bus (enseignant)

### 📊 Tableau récapitulatif

| Critère | UART | I2C | SPI | SMBus |
|---|---|---|---|---|
| **Fils** | 2-4 | 2 | 4 | 2 |
| **Débit max** | 4.5 Mbit/s | 1 Mbit/s | 18 Mbit/s | 100 kbit/s |
| **Full-duplex** | ✅ | ❌ | ✅ | ❌ |
| **Adressage** | ❌ | ✅ | ❌ | ✅ |
| **Multi-maître** | ❌ | ✅ | ❌ | ✅ |
| **Horloge** | Asynchrone | Synchrone | Synchrone | Synchrone |
| **PEC/CRC** | Optionnel | ❌ | ❌ | ✅ |
| **Distance** | Longue | Courte | Très courte | Courte |
| **Complexité** | Faible | Moyenne | Faible | Moyenne |
| **Applications** | Console, GPS | Capteurs, EEPROM | Flash, écran | Batterie, gestion |

### 📖 Règles de choix

| Besoin | Bus recommandé |
|---|---|
| Communication longue distance | **UART (RS-232/485)** |
| Communication avec un PC | **UART** |
| Plusieurs capteurs (2 fils) | **I2C** |
| Plusieurs capteurs (débit élevé) | **SPI** |
| Gestion de batterie | **SMBus** |
| Bus partagé multi-maître | **I2C / SMBus** |
| Écran graphique | **SPI** |
| GPS, modules Bluetooth | **UART** |

### 🐍 Comparaison des débits (10 min)

In [ ]:
# ============================================================
# Comparaison des 4 bus : temps de transfert
# ============================================================

def comparer_bus(taille_octets=1024):
    """
    Calcule le temps de transfert pour 1 Ko sur les 4 bus.
    """
    bus = [
        # Nom, vitesse (Hz), bits par octet (avec overhead)
        ("UART 115200",  115200,   10),   # 8N1 = 10 bits
        ("UART 921600",  921600,   10),
        ("I2C Standard",  100000,   9),   # 8 + ACK
        ("I2C Fast",      400000,   9),
        ("SPI 4 MHz",    4000000,   8),
        ("SPI 18 MHz",  18000000,   8),
        ("SMBus",        100000,   10),   # 8 + ACK + PEC occasionnel
    ]
    resultats = []
    for nom, f, bits_octet in bus:
        t_s = taille_octets * bits_octet / f
        resultats.append((nom, f, t_s))
    return resultats

TAILLE = 1024
res = comparer_bus(TAILLE)

print(f"⏱️  Transfert de {TAILLE} octets sur différents bus\n")
print(f"{'Bus':<18}{'Vitesse':<14}{'Temps':<15}{'Relatif'}")
print("-" * 60)

t_ref = max(t for _, _, t in res)
for nom, f, t in res:
    barre = "█" * int(t / t_ref * 30)
    print(f"{nom:<18}{f/1e6:>6.2f} MHz   {t*1000:>8.3f} ms   {barre}")

print(f"\n📌 À retenir :")
print(f"  → SPI 18 MHz = 120× plus rapide que SMBus.")
print(f"  → UART 921600 = 15× plus rapide que SMBus.")

---

## 🔹 QCM formatif UART / SMBus (10 min)

**1. UART est un bus :**  
A. Synchrone  
B. Asynchrone  
C. Full-duplex uniquement  
D. Multi-maître

**2. La trame UART 8N1 contient :**  
A. 8 data + 1 start + 1 stop  
B. 8 data + 1 stop  
C. 8 data + 1 start + 1 stop + 1 parity  
D. 8 data + 1 start + 1 stop, sans parité

**3. Pour F_clk = 72 MHz et baud = 115200, USART_BRR vaut environ :**  
A. 0x271  
B. 0x3E8  
C. 0x138  
D. 0x2710

**4. La principale différence entre SMBus et I2C est :**  
A. Le nombre de fils  
B. Le timeout obligatoire et le PEC  
C. La vitesse  
D. L'adressage

**5. Le PEC SMBus utilise un CRC de :**  
A. 4 bits  
B. 8 bits  
C. 16 bits  
D. 32 bits

**6. Sur le STM32F103C6T6, USART2 est sur les broches :**  
A. PA9/PA10  
B. PA2/PA3  
C. PB6/PB7  
D. PA5/PA6

**7. Le signal RTS sert à :**  
A. Envoyer les données  
B. Recevoir les données  
C. Indiquer que l'émetteur est prêt à envoyer  
D. Reset

**8. SMBus est principalement utilisé pour :**  
A. Les écrans OLED  
B. La gestion de batterie et de systèmes  
C. Les capteurs de distance  
D. Les moteurs

### ✅ Corrigé du QCM

| Q | Rép. | Justification |
|---|---|---|
| 1 | **B — Asynchrone** | UART = pas d'horloge partagée |
| 2 | **D — 8N1 sans parité** | 8N1 = 8 data, no parity, 1 stop |
| 3 | **A — 0x271** | Mantisse 39, fraction 1 |
| 4 | **B — Timeout + PEC** | Différences majeures SMBus |
| 5 | **B — 8 bits** | CRC-8 |
| 6 | **B — PA2/PA3** | USART2_TX/RX |
| 7 | **C — Émetteur prêt** | Request To Send |
| 8 | **B — Gestion batterie** | Applications typiques |

**Mon score : ___ / 8**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP11 — UART ↔ PC + démonstration SMBus

### 🎯 Objectif
Établir une communication bidirectionnelle entre le STM32 et un PC via UART2 (115200 bauds), et démontrer SMBus via I2C1.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer USART2 (PA2 TX, PA3 RX) à 115200 8N1 | 15 min | Capture CubeMX |
| 2 | Envoyer "Hello STM32" vers le PC | 10 min | Capture terminal |
| 3 | Recevoir des commandes du PC (LED ON/OFF) | 20 min | Démo |
| 4 | Implémenter un écho simple | 10 min | Code |
| 5 | Implémenter un menu interactif | 20 min | Code |
| 6 | Démontrer SMBus via I2C1 (PEC, timeout) | 10 min | Code |
| 7 | Rédiger le compte-rendu | 5 min | CR |

### ⚙️ Code — Communication UART bidirectionnelle

In [ ]:
/* ============================================================
   TP11 - Communication UART bidirectionnelle
   - PA2 : USART2_TX (vers RX du PC)
   - PA3 : USART2_RX (depuis TX du PC)
   - PC13 : LED commandée
   ============================================================ */

#include "main.h"
#include <string.h>

UART_HandleTypeDef huart2;

/* --- Envoi d'une chaîne --- */
static void uart_printf(const char *msg)
{
    HAL_UART_Transmit(&huart2, (uint8_t*)msg, strlen(msg), 100);
}

/* --- Affichage du menu --- */
static void afficher_menu(void)
{
    uart_printf("\r\n===== Menu STM32 =====\r\n");
    uart_printf("  L : Allumer LED\r\n");
    uart_printf("  E : Eteindre LED\r\n");
    uart_printf("  T : Toggle LED\r\n");
    uart_printf("  H : Afficher menu\r\n");
    uart_printf("======================\r\n");
}

/* --- Traitement d'une commande --- */
static void traiter_commande(char cmd)
{
    switch (cmd)
    {
        case 'L': case 'l':
            HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_RESET);  // actif bas
            uart_printf("LED allumee.\r\n");
            break;
        case 'E': case 'e':
            HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);
            uart_printf("LED eteinte.\r\n");
            break;
        case 'T': case 't':
            HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
            uart_printf("LED toggle.\r\n");
            break;
        case 'H': case 'h': case '?':
            afficher_menu();
            break;
        case '\r': case '\n':
            break;  // ignorer retours chariot
        default:
            uart_printf("Commande inconnue. Tapez H pour l'aide.\r\n");
            break;
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_USART2_UART_Init();

    uart_printf("\r\n\r\n=== STM32 UART Demo ===\r\n");
    afficher_menu();

    uint8_t rx_byte;

    while (1)
    {
        // Réception bloquante 1 octet (timeout 1000 ms)
        if (HAL_UART_Receive(&huart2, &rx_byte, 1, 1000) == HAL_OK)
        {
            traiter_commande((char)rx_byte);
        }
        else
        {
            // Timeout : envoyer un heartbeat toutes les secondes
            uart_printf("[heartbeat]\r\n");
        }
    }
}

### ⚙️ Code — UART avec interruption (plus efficace)

In [ ]:
/* ============================================================
   UART avec réception par interruption
   ============================================================ */

#include "main.h"

UART_HandleTypeDef huart2;

volatile uint8_t rx_byte;
volatile uint8_t rx_pret = 0;

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_USART2_UART_Init();

    // Démarrer la réception en interruption (1 octet)
    HAL_UART_Receive_IT(&huart2, (uint8_t*)&rx_byte, 1);

    while (1)
    {
        if (rx_pret)
        {
            rx_pret = 0;
            traiter_commande(rx_byte);
            // Relancer la réception pour l'octet suivant
            HAL_UART_Receive_IT(&huart2, (uint8_t*)&rx_byte, 1);
        }
        // Faire autre chose ici...
    }
}

/* --- Callback de réception --- */
void HAL_UART_RxCpltCallback(UART_HandleTypeDef *huart)
{
    if (huart->Instance == USART2)
    {
        rx_pret = 1;
    }
}

### 🔍 Analyse des codes

**Version polling :**
| Élément | Rôle |
|---|---|
| `HAL_UART_Transmit()` | Envoi bloquant |
| `HAL_UART_Receive()` | Réception avec timeout |
| Timeout 1000 ms | Utilisé pour le heartbeat |

**Version interruption :**
| Élément | Rôle |
|---|---|
| `HAL_UART_Receive_IT()` | Réception non bloquante |
| `HAL_UART_RxCpltCallback` | Appelé à chaque octet reçu |
| Relance dans la boucle | Pour recevoir l'octet suivant |

### 📖 Configuration CubeMX — USART2

```
Mode : Asynchronous
Baud Rate : 115200
Word Length : 8 bits
Parity : None
Stop Bits : 1
NVIC : cocher "USART2 global interrupt"
Broches : PA2 (TX), PA3 (RX)
```

### 🐍 Simulation Python — Réception UART côté PC (15 min)

Simulons la réception UART avec pyserial (côté PC).

In [ ]:
# ============================================================
# Script côté PC : réception UART + envoi commandes
# ============================================================

def script_pc_uart():
    """
    Script à exécuter sur le PC (avec un adaptateur USB-Série).
    Nécessite : pip install pyserial
    """
    code = '''
import serial
import time

# Ouvrir le port série
ser = serial.Serial('/dev/ttyUSB0', 115200, timeout=1)
time.sleep(2)  # laisser le temps au STM32 de démarrer

print("=== Terminal STM32 UART ===")
print("Commandes : L (LED ON), E (LED OFF), T (toggle), H (aide), Q (quitter)")

while True:
    # Lire les données disponibles
    if ser.in_waiting:
        ligne = ser.readline().decode('ascii', errors='ignore').strip()
        if ligne:
            print(f"STM32: {ligne}")

    # Lire la commande utilisateur
    cmd = input("Commande> ").strip()
    if cmd.lower() == 'q':
        break
    if cmd:
        ser.write(cmd.encode('ascii') + b'\\r\\n')

ser.close()
'''
    return code

print("💻 Script Python à exécuter sur le PC :\n")
print(script_pc_uart())

print("📌 Notes :")
print("  → Sur Linux : /dev/ttyUSB0 ou /dev/ttyACM0")
print("  → Sur Windows : COM3, COM4...")
print("  → Sur macOS : /dev/tty.usbserial-XXXX")

### 🐍 Simulation Python — Analyse de trames UART (10 min)

In [ ]:
# ============================================================
# Analyse de trames UART 8N1
# ============================================================

def decoder_trame_uart(bits, parity='none'):
    """Décode une trame UART et retourne l'octet."""
    if bits[0] != 0:
        return None, "Start bit invalide"
    octet = 0
    for i in range(8):
        octet |= bits[1 + i] << i
    idx = 9
    if parity != 'none':
        p_bit = bits[idx]
        attendu = 1 - (sum(bits[1:9]) % 2) if parity == 'odd' else sum(bits[1:9]) % 2
        if p_bit != attendu:
            return octet, f"Erreur de parité (reçu={p_bit}, attendu={attendu})"
        idx += 1
    if bits[idx] != 1:
        return octet, "Stop bit invalide"
    return octet, "OK"

# Trame correcte : 'A' = 0x41 = 0b01000001
# Bits : start=0, D0..D7 = 1,0,0,0,0,0,1,0, stop=1
trame_ok    = [0, 1, 0, 0, 0, 0, 0, 1, 0, 1]
trame_bad_start = [1, 1, 0, 0, 0, 0, 0, 1, 0, 1]
trame_bad_stop  = [0, 1, 0, 0, 0, 0, 0, 1, 0, 0]

for nom, trame in [("Trame correcte", trame_ok),
                   ("Start invalide", trame_bad_start),
                   ("Stop invalide", trame_bad_stop)]:
    octet, statut = decoder_trame_uart(trame)
    print(f"📡 {nom} :")
    print(f"   Bits   : {''.join(map(str, trame))}")
    print(f"   Octet  : 0x{octet:02X} ('{chr(octet)}')   Statut : {statut}\n")

### 📝 Compte-rendu de TP11

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- USART2 : baud = ... , format = ...
- Broches : TX = ... , RX = ...
- NVIC : ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Test de communication**
- Message envoyé depuis STM32 : ...
- Terminal PC utilisé : ...
- Commande testée : ...
- Résultat : OK / KO

**4. Menu interactif**
- Commandes implémentées : ...
- Réponses correctes ? ...
- Latence observée : ... ms

**5. Version polling vs interruption**
- CPU occupé en polling : ... %
- Réactivité en interruption : ... ms
- Conclusion : ...

**6. Démonstration SMBus**
- Calcul PEC : ...
- Adresse testée : 0x...
- Résultat : ...

**7. Problèmes rencontrés**
- ...

**8. Solutions apportées**
- ...

### 🧪 Exercice bonus — Communication UART entre deux STM32

Connecter deux cartes STM32F103C6T6 par UART (TX ↔ RX, GND ↔ GND) et échanger des messages.

**Cahier des charges :**
- Carte A : envoie "PING" toutes les secondes
- Carte B : répond "PONG" à chaque PING
- Carte A affiche le compteur de PONG reçus
- Bonus : ajouter un checksum (XOR de tous les octets)

In [ ]:
// Squelette solution bonus — Carte A (maître)

char buf[32];
uint8_t rx_buf[32];
uint16_t compteur_pong = 0;

while (1)
{
    // Envoyer PING
    HAL_UART_Transmit(&huart2, (uint8_t*)"PING\r\n", 6, 100);

    // Attendre réponse
    if (HAL_UART_Receive(&huart2, rx_buf, 6, 500) == HAL_OK)
    {
        if (strncmp((char*)rx_buf, "PONG", 4) == 0)
        {
            compteur_pong++;
            int n = snprintf(buf, sizeof(buf),
                             "PONG recus : %u\r\n", compteur_pong);
            HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
        }
    }
    HAL_Delay(1000);
}

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Rapport comparatif des 4 bus (45 min)

Rédiger un **rapport de 2 pages** comparant **UART, I2C, SPI et SMBus** :

1. **Tableau comparatif** (fils, débit, topologie, complexité)
2. **Applications typiques** pour chaque bus
3. **Avantages et inconvénients** de chacun
4. **Critères de choix** (débit, coût, nombre d'esclaves, distance)
5. **Exemples concrets** de projets avec le bus choisi

**Format :** PDF, 2 pages max.

### 🧩 Exercice 2 — Calculs UART (30 min)

Pour F_clk = **72 MHz** et les baud rates suivants, calculer `USART_BRR` et l'erreur.

| # | Baud | BRR (hex) | Mantisse | Fraction | Erreur % |
|---|---|---|---|---|---|
| 1 | 9600 | ? | ? | ? | ? |
| 2 | 19200 | ? | ? | ? | ? |
| 3 | 38400 | ? | ? | ? | ? |
| 4 | 57600 | ? | ? | ? | ? |
| 5 | 115200 | ? | ? | ? | ? |
| 6 | 230400 | ? | ? | ? | ? |
| 7 | 460800 | ? | ? | ? | ? |
| 8 | 921600 | ? | ? | ? | ? |
| 9 | 1000000 | ? | ? | ? | ? |
| 10 | 2000000 | ? | ? | ? | ? |

In [ ]:
# Corrigé Exercice 2

F_CLK = 72_000_000

print(f"{'Baud':<12}{'BRR':<10}{'Mantisse':<10}{'Frac':<8}{'Erreur %'}")
print("-" * 55)
for baud in [9600, 19200, 38400, 57600, 115200, 230400,
             460800, 921600, 1_000_000, 2_000_000]:
    brr, m, f, err = calcul_brr(F_CLK, baud)
    statut = "✅" if err < 2 else ("⚠️" if err < 5 else "❌")
    print(f"{baud:<12}0x{brr:04X}    {m:<10}{f:<8}{err:.4f}  {statut}")

### 🧩 Exercice 3 — Lecture du RM0008 (15 min)

Lire les chapitres **27 (USART)** et **26 (I2C/SMBus)** du RM0008 et répondre :

1. Combien d'USART possède le STM32F103C6T6 ? Sur quelles broches ?
2. Quel registre permet de configurer le baud rate ?
3. Quels sont les bits essentiels dans `USART_CR1` ?
4. Quelle est la différence entre **UART** et **USART** ?
5. Comment fonctionne le mode **SMBus** dans le STM32 ?
6. Que signifie le bit **SMBTYPE** dans `I2C_CR1` ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Simulateur CRC-8 (optionnel)

Implémenter un **simulateur SMBus complet** :
- Classe `SMBusDevice` avec adresse, registres internes
- Méthode `write(reg, data)` avec calcul du PEC
- Méthode `read(reg)` avec vérification du PEC
- Détection d'erreur si le PEC est incorrect

In [ ]:
# Corrigé — Simulateur SMBusDevice

class SMBusDevice:
    def __init__(self, adresse):
        self.adresse = adresse
        self.registres = [0] * 256

    def calcul_pec(self, octets):
        return crc8_smbus(octets)

    def write(self, registre, valeur):
        """Écrit une valeur avec vérification PEC."""
        trame = [self.adresse << 1, registre, valeur]
        pec_calcule = self.calcul_pec(trame)
        # Simulation : on considère le PEC correct
        self.registres[registre] = valeur
        return pec_calcule

    def read(self, registre, pec_recu=None):
        """Lit une valeur avec vérification PEC."""
        trame = [(self.adresse << 1) | 1, registre, self.registres[registre]]
        pec_attendu = self.calcul_pec(trame)
        if pec_recu is not None and pec_recu != pec_attendu:
            raise ValueError(f"PEC invalide : recu=0x{pec_recu:02X}, "
                             f"attendu=0x{pec_attendu:02X}")
        return self.registres[registre], pec_attendu

# Test
dev = SMBusDevice(adresse=0x50)

print("🔧 Simulateur SMBusDevice\n")

pec_w = dev.write(0x10, 0xAB)
print(f"Écriture : reg[0x10] = 0xAB  (PEC=0x{pec_w:02X})")

val, pec_r = dev.read(0x10)
print(f"Lecture  : reg[0x10] = 0x{val:02X}  (PEC=0x{pec_r:02X})")

# Test avec PEC erroné
try:
    dev.read(0x10, pec_recu=0xFF)
except ValueError as e:
    print(f"\n❌ Détection d'erreur PEC : {e}")

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 11

Coche ce que tu maîtrises.

- [ ] Je connais la différence entre UART et USART.
- [ ] Je connais le format d'une trame UART (start, data, parity, stop).
- [ ] Je sais calculer USART_BRR pour un baud donné.
- [ ] Je connais les broches USART2 du STM32F103C6T6 (PA2/PA3).
- [ ] Je connais les principales différences I2C vs SMBus.
- [ ] Je sais calculer un PEC (CRC-8 SMBus).
- [ ] Je sais configurer USART2 en CubeMX.
- [ ] Je sais utiliser `HAL_UART_Transmit()` et `HAL_UART_Receive()`.
- [ ] Je sais utiliser `HAL_UART_Receive_IT()` + callback.
- [ ] J'ai communiqué avec un PC via UART.
- [ ] J'ai implémenté un menu interactif.
- [ ] J'ai comparé polling vs interruption pour l'UART.
- [ ] J'ai rédigé mon compte-rendu de TP11.
- [ ] J'ai lu les chapitres 26 et 27 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S12 (TP noté n°1) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 4 |

---
# 📚 RESSOURCES Semaine 11

### Documents officiels
- 📄 **RM0008** — chapitre 27 (USART), chapitre 26 (I2C/SMBus)
- 📄 **Datasheet STM32F103x6** — section 2.3.14 (USART)
- 📄 **UM1850** — HAL UART documentation
- 📄 **SMBus Specification v2.0** (Intel)

### Outils
- **STM32CubeMX** — Connectivity → USART2
- **Adaptateur USB-Série** (CH340, CP2102, FT232)
- **Terminal série** : PuTTY, minicom, screen, CoolTerm
- **Python pyserial** pour scripts PC

### Vidéos
- *STM32 UART Tutorial (HAL)* — ControllersTech
- *SMBus vs I2C Explained* — YouTube

### Bonnes pratiques
- Vérifier les niveaux de tension (3.3 V vs 5 V)
- Croiser TX ↔ RX (TX de l'un vers RX de l'autre)
- GND commun obligatoire
- Utiliser le flow control (RTS/CTS) si débit élevé
- Préférer `HAL_UART_Receive_IT` à `HAL_UART_Receive` en production
- Implémenter un checksum/PEC pour les communications critiques

---

### 🔗 Passage à la semaine 12

**Prochaine séance :** Évaluation pratique Ateliers n°1  
- **Rappels** : GPIO, EXTI, TIMER, PWM
- **TP noté n°1** (1h30, en binôme) :
  - LED PC13 via TIM2 (interruption)
  - Bouton PA0 (EXTI0) : changement de fréquence
  - Bouton PA1 (EXTI1) : reset
  - PWM sur PA6 (TIM3_CH1) : intensité proportionnelle
  - Bonus : affichage UART2

**Préparation :**
- Revoir les notebooks S3 (GPIO), S4 (EXTI), S5 (TIMER), S6 (PWM).
- S'entraîner à écrire du code CubeMX + HAL rapide.
- Vérifier que la chaîne d'outils fonctionne.

---

**Fin du notebook — Semaine 11** ✨